SWELL_ KW dataset

In [ ]:
#datast https://www.kaggle.com/datasets/qiriro/swell-heart-rate-variability-hrv/data

In [ ]:
import numpy as np
import pandas as pd
from keras.models import Model
from keras.layers import Input, Dense, Conv2D, MaxPooling2D, Flatten, Dropout, LSTM, Bidirectional, concatenate, Layer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler, LabelEncoder
from keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Custom Attention Layer
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1), initializer="normal")
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1), initializer="zeros")
        super(AttentionLayer, self).build(input_shape)

    def call(self, x):
        et = K.squeeze(K.tanh(K.dot(x, self.W) + self.b), axis=-1)
        at = K.softmax(et)
        at = K.expand_dims(at, axis=-1)
        output = x * at
        return K.sum(output, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

# Data loading functions
def load_hrv_data(path):
    df = pd.read_csv(path)
    X = df.drop('condition', axis=1).values
    y = df['condition'].values
    return X, y


In [ ]:
df_check = pd.read_csv('/content/drive/MyDrive/DataSet/hrv_dataset/data/final/train.csv')
print("Column names in CSV:", df_check.columns)


In [ ]:


# Load data
X, y = load_hrv_data('/content/drive/MyDrive/DataSet/hrv_dataset/data/final/train.csv')

# Encode and normalize
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_onehot = pd.get_dummies(y_encoded).values
num_classes = y_onehot.shape[1]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_onehot, test_size=0.2, random_state=42)

# Input dimensions
feature_len = X.shape[1]

# Reshape data
train_X_cnn = X_train.reshape(-1, 1, feature_len, 1)
test_X_cnn = X_test.reshape(-1, 1, feature_len, 1)

train_X_lstm = X_train.reshape(-1, feature_len, 1)
test_X_lstm = X_test.reshape(-1, feature_len, 1)

print('Training data shape : ', train_X_cnn.shape, train_X_lstm.shape)
print('Testing data shape : ', test_X_cnn.shape, test_X_lstm.shape)
print('Output classes : ', label_encoder.classes_)

# Define Conv2D input
conv_input_layer = Input(shape=(1, feature_len, 1))

# Conv2D and Pooling
conv_layer = Conv2D(32, kernel_size=(1, 3), strides=(1, 1), padding='same', activation='relu')(conv_input_layer)
pooling_layer = MaxPooling2D(pool_size=(1, 2), strides=(1, 2), padding='same')(conv_layer)
flatten_layer = Flatten()(pooling_layer)

# LSTM Input
lstm_input_layer = Input(shape=(feature_len, 1))
lstm_layer = Bidirectional(LSTM(64, return_sequences=True))(lstm_input_layer)
attention_output = AttentionLayer()(lstm_layer)

# Merge CNN and Attention
merged = concatenate([flatten_layer, attention_output])

# Fully Connected Layers
dense_layer = Dense(64, activation='relu')(merged)
dropout_layer = Dropout(0.3)(dense_layer)
output_layer = Dense(num_classes, activation='softmax')(dropout_layer)

# Build Model
model = Model(inputs=[conv_input_layer, lstm_input_layer], outputs=output_layer)

# Compile
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Early Stopping
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Fit Model
history = model.fit(
    [train_X_cnn, train_X_lstm],
    y_train,
    batch_size=64,
    epochs=30,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate
score = model.evaluate([test_X_cnn, test_X_lstm], y_test, verbose=0)
print("Test Accuracy:", score[1])

# Predict and Confusion Matrix
y_pred = model.predict([test_X_cnn, test_X_lstm])
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix - HRV Stress Classification")
plt.show()
